# JevK5 — run it (GPU + Cloudflare tunnel)

JevK5 (allebee/jevk5, Qwen3.5-4B + LoRA) served as a TypeSafe-style `/v1/systemone` API and exposed via a temporary Cloudflare Quick Tunnel.

* **Accelerator:** GPU T4 x2
* **Internet:** ON
* Session + tunnel die after ~12h or on close. Training/testing only.


In [ ]:
import sys, subprocess
print("python", sys.version.split()[0])
subprocess.run(["nvidia-smi"], check=False)


In [ ]:
# --- install JevK5 ---
!git clone --depth 1 --branch v0.2.0 https://github.com/allebee/jevk5 /kaggle/working/jevk5
!cd /kaggle/working/jevk5 && pip -q install -e .
!pip -q install --upgrade "Pillow>=11.2"
print("installed")


In [ ]:
# --- quick sanity: import + load (downloads ~9GB first time) ---
from jevk5 import JevK5
import time
t0 = time.time()
model = JevK5("alibiserikbay/JevK5")
print(f"loaded in {time.time()-t0:.0f}s, device={model.device}")


In [ ]:
# --- test all 3 question types directly ---
state = "Double charged order #4411, refund today or we cancel"

print("choice:", model.decide(state, {"type": "choice", "instructions": "Which team?",
    "criteria": {"billing": "Payments/refunds", "tech": "Bugs", "sales": "New"}}))
print("noul  :", model.decide(state, {"type": "noul", "instructions": "Asks for money back?"}))
print("score :", model.decide(state, {"type": "score", "instructions": "How angry?",
    "criteria": ["calm", "frustrated", "very angry"]}))


In [ ]:
# --- start jevk5-serve on :8090 (GPU) ---
import threading, subprocess
def _run(cmd, cwd):
    print("start", cmd[0], flush=True)
    subprocess.run(cmd, cwd=cwd, check=True)

threading.Thread(target=_run, args=(["jevk5-serve", "--model", "alibiserikbay/JevK5",
                                     "--host", "0.0.0.0", "--port", "8090"],
                                    "/kaggle/working"), daemon=True).start()
print("serving on :8090 ...")


In [ ]:
# --- test the HTTP endpoint ---
import httpx, time
time.sleep(10)
body = {"state": "Order #7120 shows delivered to No. 17; the customer lives at No. 71.",
        "questions": {"what": {"type": "choice", "instructions": "What happened to the parcel?",
                               "criteria": ["delivered", "misdelivered", "unknown"]}}}
for _ in range(30):
    try:
        r = httpx.post("http://127.0.0.1:8090/v1/systemone", json=body, timeout=180)
        print("jevk5:", r.status_code, str(r.json())[:250])
        break
    except Exception as e:
        print("waiting...", type(e).__name__)
        time.sleep(10)


In [ ]:
# --- cloudflare tunnel -> prints PUBLIC URL ---
import subprocess, time
log = open("/kaggle/working/cf.log", "w")
p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8090"],
                     stdout=log, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(15):
    time.sleep(10)
    log.flush()
    try:
        txt = open("/kaggle/working/cf.log").read()
    except Exception:
        txt = ""
    for line in txt.splitlines():
        if "trycloudflare.com" in line:
            url = line.strip().split(" ")[-1]
            break
    if url:
        break
print("TUNNEL_URL=", url)


### Public endpoint
`POST https://<tunnel>.trycloudflare.com/v1/systemone`

```json
{"state": "Order #7120 shows delivered to No. 17; the customer lives at No. 71.",
 "questions": {"what": {"type": "choice", "instructions": "What happened to the parcel?",
                        "criteria": ["delivered", "misdelivered", "unknown"]}}}
```
Response: `{model, answers, usage, latency_ms}`. Supports `noul` / `choice` / `score` (criteria = dict or list). Keep the notebook running to keep the URL live.
